# HEA Hardness KAN — Demo

This notebook demonstrates how to use the `HEA_pyKAN` library to evaluate a
Kolmogorov–Arnold Network (KAN) model for predicting Vickers Hardness (HV) of
High Entropy Alloys (HEAs).

**Dataset:** Berry/Gorsse 2018 — 244 HEA compositions with 10 elemental
fraction inputs and HV output (range 100–905, mean ~451 HV).

**Model:** A single-hidden-layer KAN (`[10, hidden, 1]`) using spline
activations.  The hidden layer width is the main architectural variable;
the spline order `k` and grid resolution `g` are also tuned.

**Workflow:**
1. Load and z-score normalise the dataset
2. Load the best hyperparameters found by an Optuna sweep (100 trials per hidden-node count, fixed shuffle seed for reproducibility)
3. Choose a hidden-node count and run 5-fold cross-validation
4. Repeat across multiple shuffle seeds to separate model variance from data-split variance

---

**Regularisation note:**  
pyKAN's `fit()` applies regularisation as:

$$\mathcal{L}_\text{reg} = \lambda \left( \lambda_\text{l1} \cdot L_1 + \lambda_\text{entropy} \cdot H + \ldots \right)$$

so `lamb_entropy` passed to the model is *relative* to the master weight `lamb`.
The JSON stores the *absolute* entropy weight (`lamb_entropy_actual = lamb × lamb_entropy_model`),
so the conversion `lamb_entropy = params["lamb_entropy"] / params["lamb"]` is required before use.

In [ ]:
import json
import sys
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from kan import *

# ── ensure HEA_pyKAN.py (same directory as this notebook) is importable ──────
sys.path.insert(0, os.path.abspath(''))
from HEA_pyKAN import to_np, train_val_test_split, best_loss_KAN, k_fold_val

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

## 1  Load and normalise data

The raw CSV contains both composition (atomic fractions for 10 elements) and
measured HV.  Rows without an HV measurement are dropped.

All features — inputs and output — are z-score normalised to zero mean and unit
variance before training.  This is important for KAN spline fitting, which is
sensitive to input scale.  The HV standard deviation (`stdev["HV"]`) is saved
as `data_norm` so that normalised MAE values can be converted back to HV units
for reporting.

In [ ]:
# ── set your data path here ──────────────────────────────────────────────────
DATA_PATH = r"H:\My Drive\Documents and Data\Data\BerryML"
DATA_FILE = "Gorsse2018ML_Data_RawHV.csv"

df = pd.read_csv(os.path.join(DATA_PATH, DATA_FILE))
cols = list(df.columns)

# keep rows with a valid HV measurement; select the composition + HV columns
df = df[df["HV"].notna()][cols[25:]]
print(f"Dataset shape: {df.shape}")

In [ ]:
mean  = df.mean()
stdev = df.std()
norm_df = (df - mean) / stdev

x_data = torch.tensor(norm_df[cols[26:]].values, dtype=torch.float32).to(device)
y_data = torch.tensor(norm_df[cols[25]].values,  dtype=torch.float32).unsqueeze(-1).to(device)

data_norm = stdev["HV"]   # scale factor: multiply normalised MAE → HV units

print(f"x: {x_data.shape},  y: {y_data.shape}")
print(f"HV  mean={mean['HV']:.1f}  std={stdev['HV']:.1f}  "
      f"min={df['HV'].min():.0f}  max={df['HV'].max():.0f}")

## 2  Load best hyperparameters

The JSON file contains the best trial from an Optuna search (100 trials per
hidden-node count) with a fixed cross-validation shuffle seed (`seed=42`),
making the results directly comparable across hidden-node counts.

Parameters stored per hidden-node count:

| Parameter | Meaning |
|---|---|
| `k` | Spline order (1 = piecewise linear, higher = smoother) |
| `g` | Number of spline grid intervals |
| `lamb` | Master regularisation weight |
| `lamb_entropy` | **Absolute** entropy penalty weight (see regularisation note above) |
| `model_seed` | KAN weight initialisation seed |
| `cv_mae_HV` | Mean absolute error from the Optuna trial's cross-validation (HV units) |

Note that the Optuna CV MAE is a single-seed estimate; the repeated CV in
section 5 gives a more reliable picture of true generalisation performance.

In [ ]:
HYPERPARAM_FILE = os.path.join(os.path.abspath(''), "best_hyperparams_by_hidden_nodes.json")

with open(HYPERPARAM_FILE) as f:
    best_params = json.load(f)

# quick overview
print(f"{'hidden':>8}  {'k':>3}  {'g':>3}  {'lamb':>10}  {'lamb_entropy_actual':>20}  {'model_seed':>10}  {'cv_mae_HV':>10}")
print("-" * 78)
for h, res in best_params.items():
    p = res["params"]
    print(f"{h:>8}  {p['k']:>3}  {p['g']:>3}  {p['lamb']:>10.3e}  "
          f"{p['lamb_entropy']:>20.3e}  {p['model_seed']:>10}  "
          f"{res['value']:>10.1f}")

## 3  Select hidden-node count and extract parameters

Change `HIDDEN` to any of the values in the JSON (1, 2, 3, 4, 6, 8, 10, 15,
20, 25, 30) to evaluate a different model size.

The key step here is the `lamb_entropy` conversion: the JSON stores the
absolute entropy weight, but pyKAN's `fit()` expects the value *relative* to
`lamb`.  Dividing by `lamb` recovers the correct model parameter.

In [ ]:
HIDDEN = 20   # ← change this to any key in the JSON: 1,2,3,4,6,8,10,15,20,25,30

params = best_params[str(HIDDEN)]["params"]

k           = params["k"]
g           = params["g"]
lamb        = params["lamb"]
model_seed  = params["model_seed"]

# convert stored absolute entropy weight → model parameter (see module docstring)
lamb_entropy = params["lamb_entropy"] / lamb

print(f"Hidden nodes : {HIDDEN}")
print(f"k={k}, g={g}, model_seed={model_seed}")
print(f"lamb                    = {lamb:.3e}")
print(f"lamb_entropy (stored)   = {params['lamb_entropy']:.3e}  ← absolute weight")
print(f"lamb_entropy (model)    = {lamb_entropy:.3e}  ← passed to fit()")

## 4  Single cross-validation run

`k_fold_val` runs 5-fold CV with a fixed shuffle seed.  For each fold, the
held-out data is split 50:50 into test and validation sets and **two models are
trained** (A and B, swapping the test/val roles).  This gives 10 MAE
measurements per call (5 folds × 2 models), which are averaged and returned.

The `best_loss_KAN` inside `k_fold_val` monitors validation loss at each
training step and restores the checkpoint with the lowest validation loss at
the end — a form of early stopping that helps in overparameterised regimes
without terminating training early.

In [ ]:
mae, mae_error, stdev_mae = k_fold_val(
    x_data,
    y_data,
    data_norm  = data_norm,
    model_seed = model_seed,
    shuffle_seed = 42,
    splits     = 5,
    hidden     = HIDDEN,
    grid       = g,
    k          = k,
    steps      = 25,
    loss_fn    = nn.L1Loss(),
    lamb       = lamb,
    lamb_entropy = lamb_entropy,
    device     = str(device),
    verbose    = True,
)

print(f"\nHidden={HIDDEN}: MAE = {mae:.1f} ± {mae_error:.1f} HV  "
      f"(stdev across folds = {stdev_mae:.1f} HV)")

## 5  Repeated cross-validation across shuffle seeds

A single CV run can be noisy because the 5-fold split is one random
partitioning of the data.  Here we run 10 repeats, each with a different
`shuffle_seed`, to get a stable estimate of generalisation performance.

The model seed is held **fixed** across repeats so that variance comes from
the data split, not from random weight initialisation.  The reported
uncertainty (± standard error) therefore reflects how sensitive the result is
to which samples land in which fold — a useful check that the model is not
relying on a lucky partition.

In [ ]:
REPEATS = 10

maes = []
for r in range(REPEATS):
    mae_r, _, _ = k_fold_val(
        x_data,
        y_data,
        data_norm    = data_norm,
        model_seed   = model_seed,
        shuffle_seed = r,
        splits       = 5,
        hidden       = HIDDEN,
        grid         = g,
        k            = k,
        steps        = 25,
        loss_fn      = nn.L1Loss(),
        lamb         = lamb,
        lamb_entropy = lamb_entropy,
        device       = str(device),
        verbose      = False,
    )
    maes.append(mae_r)
    print(f"  repeat {r:>2}: {mae_r:.1f} HV")

maes = np.array(maes)
print(f"\nMean over {REPEATS} repeats: {maes.mean():.1f} ± {maes.std(ddof=1)/np.sqrt(REPEATS):.1f} HV")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.bar(range(REPEATS), maes, color="steelblue", alpha=0.8)
ax.axhline(maes.mean(), color="k", lw=1.5, linestyle="--", label=f"mean = {maes.mean():.1f} HV")
ax.set_xlabel("Repeat (shuffle seed)")
ax.set_ylabel("MAE (HV)")
ax.set_title(f"5-fold CV repeated {REPEATS}×  |  hidden={HIDDEN}")
ax.legend()
plt.tight_layout()
plt.show()